# Day 16 · 数据摄取

> **前置**: Day11-15 已掌握 NumPy / Pandas / 可视化  
> **关键问题**: 数据从哪里来?CSV、JSON、数据库是最常见的三种源头,本节学会用 Pandas 把它们全部读进 DataFrame,再把处理结果导回去。

**引入(5 分钟)**

上一节学会了"看"数据(`df.head()` / `df.describe()`),这一节学"取"数据。真实项目中,数据可能是一个 CSV 导出文件、一个接口返回的 
JSON、一张数据库表。今天的任务就是掌握 **读入** 与 **导出** 两套工具链。


**1. read_csv 参数详解**

**read_csv** 是最常用的读入函数,几乎每个项目都会遇到。本节重点掌握 **encoding**(文件编码,中文 CSV 常用 utf-8 或 
gbk)、**sep**(分隔符,默认逗号,制表符用 `\t`)、**header**(第几行做列名)、**index_col**(指定某列做行索引)、**usecols**(只读部分列,节省
内存)、**nrows**(只读前 N 行)、**skiprows**(跳过若干行)、**na_values**(哪些值视为缺失值)。


In [ ]:
import pandas as pd

# 用硬编码 DataFrame 模拟一份学生数据
df_demo = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六"],
    "年龄": [20, 21, 19, 22],
    "城市": ["北京", "上海", "广州", "深圳"],
})
print("--- 原始 DataFrame ---")
print(df_demo)

# 先把它写出到临时 CSV,再用 read_csv 读回
# 真实项目中这里的 "students.csv" 就是真实文件路径
df_demo.to_csv("students.csv", index=False,
               encoding="utf-8")
df = pd.read_csv("students.csv", encoding="utf-8")
print("\n--- read_csv 读回 ---")
print(df)

下面演示 **sep**、**index_col**、**header** 三个参数,分别设置分隔符、行索引、列名来源。

In [ ]:
import pandas as pd

# sep 参数:以分号做分隔符
df_sep = pd.DataFrame({
    "姓名": ["张三", "李四"],
    "年龄": [20, 21],
})
df_sep.to_csv("students_sep.csv", index=False,
              sep=";", encoding="utf-8")

# 读取时必须指定 sep=";",否则整行都会变成一列
df = pd.read_csv("students_sep.csv", sep=";",
                 encoding="utf-8")
print("--- sep=';' ---")
print(df)

# index_col 参数:把第 0 列(姓名)设为行索引
df_idx = pd.read_csv("students.csv",
                     index_col=0, encoding="utf-8")
print("\n--- index_col=0 ---")
print(df_idx)

# header 参数:指定第几行做列名(此处等同默认)
df_head = pd.read_csv("students.csv",
                      header=0, encoding="utf-8")
print("\n--- header=0(默认) ---")
print(df_head)

接下来继续演示 **usecols**、**nrows**、**skiprows**、**na_values** 四个实用参数。

In [ ]:
import pandas as pd

# 构造含缺失值的 CSV
df_na = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六"],
    "年龄": [20, None, 19, 22],
    "成绩": [88.5, 92.0, None, 76.5],
})
df_na.to_csv("students_na.csv", index=False,
             encoding="utf-8")

# usecols:只取姓名和成绩两列,省内存
df_cols = pd.read_csv("students_na.csv",
                      usecols=["姓名", "成绩"],
                      encoding="utf-8")
print("--- usecols ---")
print(df_cols)

# nrows:只读前 2 行(常用于预览大文件)
df_n = pd.read_csv("students_na.csv",
                   nrows=2, encoding="utf-8")
print("\n--- nrows=2 ---")
print(df_n)

# skiprows:跳过第 2 行(索引为 1 的行)
df_skip = pd.read_csv("students_na.csv",
                      skiprows=[1], encoding="utf-8")
print("\n--- skiprows=[1] ---")
print(df_skip)

# na_values:把 "NA" 和 "-" 也视为 NaN
df_na2 = pd.DataFrame({
    "姓名": ["张三", "李四", "王五"],
    "年龄": [20, "NA", "-"],
})
df_na2.to_csv("students_na2.csv", index=False,
              encoding="utf-8")
df_na_vals = pd.read_csv("students_na2.csv",
                         na_values=["NA", "-"],
                         encoding="utf-8")
print("\n--- na_values=[\"NA\",\"-\"] ---")
print(df_na_vals)

**✏️ 练一练 ⭐**

下面的字典是一份成绩单 CSV,其中成绩列混入了 `"缺考"` 字符串。请用 `read_csv` 把它读入并要求:1) 只读取"姓名"和"成绩"两列;2) 把 `"缺考"` 
视为缺失值;3) 打印 `df.isna()` 查看缺失情况。


In [ ]:
import pandas as pd

# 构造 CSV 文件
df_score = pd.DataFrame({
    "姓名": ["小明", "小红", "小刚", "小丽"],
    "语文": [88, 92, 76, 85],
    "数学": [95, "缺考", 88, 79],
    "英语": [80, 90, "缺考", 86],
})
df_score.to_csv("score.csv", index=False,
                encoding="utf-8")

# =====================
# 学员代码区
# =====================
pass

# =====================
# 参考答案
# =====================
# df = pd.read_csv(
#     "score.csv",
#     usecols=["姓名", "数学"],
#     na_values=["缺考"],
#     encoding="utf-8",
# )
# print("--- 读入结果 ---")
# print(df)
# print("\n--- 缺失值检查 ---")
# print(df.isna())

**2. read_json**

**read_json** 支持 JSON 字符串或 JSON 文件路径。最常用的是 **orient="records"**(列表套字典)。中文场景导出时务必加 
**force_ascii=False**,否则中文会被转义成 `\uXXXX`。


In [ ]:
import pandas as pd
import json

# 利用 DataFrame 构造 JSON 字符串(orient="records")
df_j = pd.DataFrame({
    "姓名": ["张三", "李四", "王五"],
    "年龄": [20, 21, 19],
})
json_str = df_j.to_json(orient="records",
                        force_ascii=False)
print("--- 导出的 JSON 字符串 ---")
print(json_str)

# 再把 JSON 字符串读回 DataFrame
df_back = pd.read_json(json_str)
print("\n--- read_json 读回 ---")
print(df_back)

# 写成 JSON 文件后再读取
df_j.to_json("data.json", orient="records",
             force_ascii=False, indent=2)
df_file = pd.read_json("data.json")
print("\n--- 从文件读 JSON ---")
print(df_file)

下面演示 **orient** 参数的三种取值: `"records"`(列表套字典)、`"index"`(字典套字典,键是索引)、`"columns"`(外层键是列名)。

In [ ]:
import pandas as pd

df_o = pd.DataFrame({
    "姓名": ["张三", "李四"],
    "年龄": [20, 21],
}, index=["a", "b"])

# orient="records":最通用,列表套字典
print('--- orient="records" ---')
print(df_o.to_json(orient="records",
                   force_ascii=False))

# orient="index":外层键是行索引
print('\n--- orient="index" ---')
print(df_o.to_json(orient="index",
                   force_ascii=False))

# orient="columns":外层键是列名
print('\n--- orient="columns" ---')
print(df_o.to_json(orient="columns",
                   force_ascii=False))

**✏️ 练一练 ⭐⭐**

下面是一份图书馆藏字典,请用 `pd.DataFrame()` 把它转成 DataFrame,再用 `to_json()` 导出为 `orient="records"` 格式,最后写回 
`"books.json"` 文件。


In [ ]:
import pandas as pd

# 原始数据(列表套字典)
data = [
    {"书名": "Python编程", "作者": "小王", "价格": 59.8},
    {"书名": "数据分析", "作者": "小李", "价格": 88.0},
    {"书名": "机器学习", "作者": "小张", "价格": 102.5},
]

# =====================
# 学员代码区
# =====================
pass

# =====================
# 参考答案
# =====================
# df = pd.DataFrame(data)
# df.to_json("books.json", orient="records",
#            force_ascii=False, indent=2)
# # 验证
# df_check = pd.read_json("books.json")
# print(df_check)

**3. read_sql (SQLite)**

**read_sql** 配合 SQLite 数据库使用。标准流程: **sqlite3.connect()** 建立连接 → 创建表、插入数据 → 
**pd.read_sql(query, conn)** 把查询结果读成 DataFrame → 最后用 **conn.close()** 关闭连接,或者用 **with** 
上下文管理器自动关闭。


In [ ]:
import pandas as pd
import sqlite3

# 创建内存数据库(程序结束自动销毁)
conn = sqlite3.connect(":memory:")
cur = conn.cursor()

# 创建 students 表
cur.execute(
    "CREATE TABLE students "
    "(id INTEGER PRIMARY KEY, name TEXT, age INTEGER)"
)

# 批量插入数据
rows = [(1, "张三", 20), (2, "李四", 21),
        (3, "王五", 19), (4, "赵六", 22)]
cur.executemany(
    "INSERT INTO students VALUES (?, ?, ?)", rows
)
conn.commit()

# 条件查询 age>20,读成 DataFrame
df_sql = pd.read_sql(
    "SELECT * FROM students WHERE age > 20", conn
)
print("--- 条件查询 age>20 ---")
print(df_sql)

# 关闭连接(必须调用,否则可能锁表)
conn.close()

上面手动调用了 `conn.close()`。实际项目中推荐 **with** 上下文管理,代码更简洁,不会遗漏关闭。

In [ ]:
import pandas as pd
import sqlite3

# 用 with 上下文管理器(更安全)
with sqlite3.connect(":memory:") as conn2:
    cur2 = conn2.cursor()
    cur2.execute(
        "CREATE TABLE books "
        "(title TEXT, price REAL)"
    )
    cur2.executemany(
        "INSERT INTO books VALUES (?, ?)",
        [("Python基础", 59.8),
         ("数据分析", 88.0)],
    )
    conn2.commit()
    df_books = pd.read_sql(
        "SELECT * FROM books", conn2
    )
    print("--- with 上下文读取 books ---")
    print(df_books)
# with 块结束,conn2 自动关闭

**✏️ 练一练 ⭐⭐**

请在内存 SQLite 中创建 `scores` 表,字段: `id`、`name`、`score`,插入 4 条记录,然后用 `read_sql` 查询并打印 `score >= 60` 
的及格名单。


In [ ]:
import pandas as pd
import sqlite3

# =====================
# 学员代码区
# =====================
pass

# =====================
# 参考答案
# =====================
# conn = sqlite3.connect(":memory:")
# cur = conn.cursor()
# cur.execute(
#     "CREATE TABLE scores "
#     "(id INTEGER, name TEXT, score INTEGER)"
# )
# rows = [(1, "小明", 85), (2, "小红", 55),
#         (3, "小刚", 92), (4, "小丽", 60)]
# cur.executemany(
#     "INSERT INTO scores VALUES (?, ?, ?)", rows
# )
# conn.commit()
# df = pd.read_sql(
#     "SELECT * FROM scores WHERE score >= 60", conn
# )
# print("--- 及格名单 ---")
# print(df)
# conn.close()

**4. 数据导出 to_csv**

分析完成后常需把结果持久化。**to_csv** 默认会把行索引也写成单独一列,通常不需要,所以必须加 **index=True**,并指定 **encoding** 防止中文乱码。

In [ ]:
import pandas as pd

# 构造一份数据
df_out = pd.DataFrame({
    "姓名": ["张三", "李四", "王五"],
    "年龄": [20, 21, 19],
    "成绩": [88.5, 92.0, 76.5],
})

# 默认导出:会多出一列行索引(0,1,2)
df_out.to_csv("with_index.csv",
              encoding="utf-8")
print("--- 默认导出(含行索引) ---")
print(pd.read_csv("with_index.csv"))

# index=False:去掉行索引,更干净
df_out.to_csv("no_index.csv", index=False,
              encoding="utf-8")
print("\n--- index=False 导出 ---")
print(pd.read_csv("no_index.csv"))

**✏️ 练一练 ⭐**

把下面这份"销售报表"导出为 `"sales.csv"`,要求:1) 不要行索引;2) 编码为 utf-8;3) 导出后用 `read_csv` 读回并验证。

In [ ]:
import pandas as pd

df_sales = pd.DataFrame({
    "月份": ["1月", "2月", "3月"],
    "销售额": [12000, 15800, 13400],
})

# =====================
# 学员代码区
# =====================
pass

# =====================
# 参考答案
# =====================
# df_sales.to_csv("sales.csv", index=False,
#                encoding="utf-8")
# df_check = pd.read_csv("sales.csv",
#                        encoding="utf-8")
# print(df_check)

**5. 数据导出 to_json**

**to_json** 最常用的是 **orient="records"**(列表套字典,其他语言读起来最方便),配合 **force_ascii=False** 保留中文,加 
**indent=2** 美化格式。


In [ ]:
import pandas as pd

df_j = pd.DataFrame({
    "姓名": ["张三", "李四"],
    "年龄": [20, 21],
})

# orient="records" + force_ascii=False + indent=2
df_j.to_json("pretty.json", orient="records",
             force_ascii=False, indent=2)
print("--- 已写出 pretty.json ---")

# 读回验证
df_back = pd.read_json("pretty.json")
print(df_back)

# 对比:force_ascii=True(默认)会把中文转义
print("\n--- force_ascii=True(默认) ---")
print(df_j.to_json(orient="records"))

print("\n--- force_ascii=False ---")
print(df_j.to_json(orient="records",
                   force_ascii=False))

**✏️ 练一练 ⭐⭐**

把下面这份"员工表"导出为 `"staff.json"`,要求使用 `orient="records"`、`force_ascii=False`、`indent=2`,然后读回并打印。

In [ ]:
import pandas as pd

df_staff = pd.DataFrame({
    "工号": [101, 102, 103],
    "姓名": ["刘备", "关羽", "张飞"],
    "部门": ["总办", "军师", "先锋"],
})

# =====================
# 学员代码区
# =====================
pass

# =====================
# 参考答案
# =====================
# df_staff.to_json(
#     "staff.json", orient="records",
#     force_ascii=False, indent=2
# )
# df_check = pd.read_json("staff.json")
# print(df_check)

**6. 综合练习:数据库 → DataFrame → 导出**

把今天的所有知识点串起来:从 SQLite 读取两张表 → Pandas 合并 → 导出为 CSV 和 JSON。这是真实项目里最常见的流水线。

In [ ]:
import pandas as pd
import sqlite3

# 1. 在内存数据库建表并写入数据
with sqlite3.connect(":memory:") as conn:
    cur = conn.cursor()
    # 学生表
    cur.execute(
        "CREATE TABLE stu "
        "(id INTEGER, name TEXT, class_id INTEGER)"
    )
    cur.executemany(
        "INSERT INTO stu VALUES (?, ?, ?)",
        [(1, "张三", 1), (2, "李四", 1),
         (3, "王五", 2)],
    )
    # 班级表
    cur.execute(
        "CREATE TABLE cls "
        "(class_id INTEGER, class_name TEXT)"
    )
    cur.executemany(
        "INSERT INTO cls VALUES (?, ?)",
        [(1, "一班"), (2, "二班")],
    )
    conn.commit()

    # 2. 跨表查询(用字符串写出 SQL)
    sql = (
        "SELECT stu.name, cls.class_name "
        "FROM stu JOIN cls "
        "ON stu.class_id = cls.class_id"
    )
    df_all = pd.read_sql(sql, conn)
    print("--- 跨表查询结果 ---")
    print(df_all)

# 3. 导出 CSV
df_all.to_csv("class_info.csv", index=False,
              encoding="utf-8")
print("\n--- 已导出 class_info.csv ---")

# 4. 导出 JSON
df_all.to_json("class_info.json", orient="records",
               force_ascii=False, indent=2)
print("--- 已导出 class_info.json ---")

**✏️ 练一练 ⭐⭐⭐**

请在内存 SQLite 中创建 `orders`(订单表) 和 `products`(商品表),用 `read_sql` 做 JOIN 查询,将"订单号 + 商品名 + 金额"的结果导出为 
`"order_report.csv"`(不要行索引)。数据自己设计,每张表至少 3 行。


In [ ]:
import pandas as pd
import sqlite3

# =====================
# 学员代码区
# =====================
pass

# =====================
# 参考答案
# =====================
# with sqlite3.connect(":memory:") as conn:
#     cur = conn.cursor()
#     cur.execute(
#         "CREATE TABLE products "
#         "(pid INTEGER, pname TEXT, price REAL)"
#     )
#     cur.executemany(
#         "INSERT INTO products VALUES (?, ?, ?)",
#         [(1, "键盘", 199), (2, "鼠标", 89),
#          (3, "显示器", 1299)],
#     )
#     cur.execute(
#         "CREATE TABLE orders "
#         "(oid INTEGER, pid INTEGER, qty INTEGER)"
#     )
#     cur.executemany(
#         "INSERT INTO orders VALUES (?, ?, ?)",
#         [(1001, 1, 2), (1002, 3, 1),
#          (1003, 2, 5)],
#     )
#     conn.commit()
#     sql = (
#         "SELECT orders.oid, products.pname, "
#         "products.price * orders.qty AS total "
#         "FROM orders JOIN products "
#         "ON orders.pid = products.pid"
#     )
#     df = pd.read_sql(sql, conn)
#     df.to_csv("order_report.csv", index=False,
#               encoding="utf-8")
#     print("\u5df2\u5bfc\u51fa order_report.csv")
#     print(df)

**📝 本节试题集**

当堂练:`course/lesson16/in_class/practice01-06.py`(6 道)  
课后作业:`course/lesson16/homework/task01-03.py`(3 道)

**今日小结**

| 函数 | 方向 | 关键参数 |
| --- | --- | --- |
| **pd.read_csv** | 读 CSV | encoding / sep / header / index_col / usecols / nrows / skiprows / 
na_values |
| **pd.read_json** | 读 JSON | orient(records 最常用) |
| **pd.read_sql** | 读 SQL | sql 语句 + conn 连接对象 |
| **df.to_csv** | 写 CSV | index=False / encoding |
| **df.to_json** | 写 JSON | orient="records" / force_ascii=False |

**易错点速记**

1. **中文 CSV 乱码**: 默认编码 utf-8 不行时,尝试 `encoding="gbk"`。
2. **数据库连接未关闭**: 读/写完务必 `conn.close()`,推荐 `with` 上下文。
3. **JSON 中文被转义**: `to_json()` 一定要加 `force_ascii=False`。
4. **to_csv 多出一列序号**: 忘加 `index=False`。
5. **na_values 没生效**: 确认 CSV 里的字符串和列表里的完全一致(比如空格)。